# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [4]:
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv'

df = pd.read_csv(DATA_URL)

df.columns = df.columns.str.lower().str.replace(' ', '_')

print("--- DataFrame Columns after Standardization ---")
print(df.columns.tolist())
print("-" * 50)

--- DataFrame Columns after Standardization ---
['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type']
--------------------------------------------------


In [6]:
df_low_claim_respondents = df[
    (df['total_claim_amount'] < 1000) &
    (df['response'] == 'Yes')
].copy() # .copy() is used to avoid SettingWithCopyWarning

print("Task 1: New DataFrame")
print(f"New DataFrame shape: {df_low_claim_respondents.shape}")
print("First 5 rows of the filtered DataFrame:\n", df_low_claim_respondents.head())
print("\n" + "="*50 + "\n")

#2
df_respondents = df[df['response'] == 'Yes']


df_respondents = df[df['response'] == 'Yes']

analysis_2 = df_respondents.groupby(['policy_type', 'gender']).agg(
    avg_monthly_premium=('monthly_premium_auto', 'mean'),
    avg_clv=('customer_lifetime_value', 'mean'),
    avg_total_claim_amount=('total_claim_amount', 'mean'),
    customer_count=('customer', 'count')
).sort_values(by='avg_clv', ascending=False).round(2)

print("--- Task 2: Respondent Analysis by Policy Type and Gender ---")
print(analysis_2)

print("\n--- Task 2: Profitability and Risk Discussion ---")
print("1. Most Profitable/Low-Risk Segments (High CLV & Low Claims):")
print("   - Look for high 'avg_clv' AND low 'avg_total_claim_amount'.")
print(f"   - **Example:** Segments with the highest CLV ({analysis_2.index[0][0]}, {analysis_2.index[0][1]}) should be examined. If their 'avg_total_claim_amount' is low relative to other groups, they are highly profitable and low-risk.")
print("\n2. High-Risk Segments (High Claims):")
print("   - Look for the highest 'avg_total_claim_amount'. These segments pose the highest risk.")
highest_claim_group = analysis_2['avg_total_claim_amount'].idxmax()
print(f"   - **Example:** The highest average total claim amount is for {highest_claim_group[0]} ({highest_claim_group[1]}). This group may be high-risk, despite their CLV.")
print("\n" + "="*50 + "\n")

# 3

# 3a
state_counts_analysis = df['state'].value_counts().rename('customer_count').reset_index()
state_counts_analysis.columns = ['state', 'customer_count']

# 3b
filtered_states = state_counts_analysis[state_counts_analysis['customer_count'] > 500].sort_values(by='customer_count', ascending=False)

print("--- Task 3: State Segmentation (Total Customers) ---")
print("Total customers per state (All States):\n", state_counts_analysis)
print("\nFiltered States (More than 500 Customers):\n", filtered_states)
print("\n" + "="*50 + "\n")

# 4
analysis_4 = df.groupby('education')['customer_lifetime_value'].agg(
    min_clv='min',
    max_clv='max',
    median_clv='median'
).sort_values(by='median_clv', ascending=False).round(2)

print("--- Task 4: CLV Summary Statistics by Education Level ---")
print(analysis_4)

# Conclusions:
print("\nTask 4: Conclusions on CLV")
print("1. Median CLV Comparison: The median CLV is highest for Doctor and lowest for High School or Below, suggesting that customers with higher educational attainment are generally more valuable.")
print("2. Variability (Max-Min): The range (Max CLV - Min CLV) is vast across all groups, indicating a high degree of variability in customer value regardless of education. For example, the highest max CLV belongs to the **Bachelor** group, showing potential for very high-value customers within this segment.")
print("3. Focus: To make returns bigger, the company might focus acquisition efforts on Doctor and Master segments due to their consistently higher median value.")

--- Task 1: New DataFrame (Low Claims & Respondents) ---
New DataFrame shape: (1399, 26)
First 5 rows of the filtered DataFrame:
     unnamed:_0 customer       state  customer_lifetime_value response  \
3            3  XL78013      Oregon             22332.439460      Yes   
8            8  FM55990  California              5989.773931      Yes   
15          15  CW49887  California              4626.801093      Yes   
19          19  NJ54277  California              3746.751625      Yes   
27          27  MQ68407      Oregon              4376.363592      Yes   

    coverage education effective_to_date employmentstatus gender  ...  \
3   Extended   College           1/11/11         Employed      M  ...   
8    Premium   College           1/19/11         Employed      M  ...   
15     Basic    Master           1/16/11         Employed      F  ...   
19  Extended   College           2/26/11         Employed      F  ...   
27   Premium  Bachelor           2/28/11         Employed      F  

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here